In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
!pip install openai-whisper

In [ ]:
import whisper

model = whisper.load_model("small")  # VERY IMPORTANT LINE

In [ ]:
model = model.to("cuda")

In [ ]:
import requests

url = "https://storage.googleapis.com/upload_goai/967179/825780_transcription.json"
data = requests.get(url).json()

print("Loaded:", len(data))
print(data[0])

In [ ]:
print(type(data))
print(len(data))

print("Sample keys:", data[0].keys())
print("Full sample:", data[0])

In [ ]:
print(data[0])

In [ ]:
{
  "audio": "...",
  "text": "..."
}

In [ ]:
{
  "file_path": "...",
  "transcription": "..."
}

In [ ]:
print(data[0])

In [ ]:
data[0]

In [ ]:
print(data[:3])

In [ ]:
!pip install datasets transformers jiwer librosa torchaudio openai-whisper

In [ ]:
!pip install datasets openai-whisper jiwer soundfile

In [ ]:
from datasets import load_dataset

dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

In [ ]:
print(dataset[0])

In [ ]:
import whisper
import soundfile as sf

model = whisper.load_model("small").to("cuda")

sample = dataset[0]

audio = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]

sf.write("sample.wav", audio, sr)

result = model.transcribe("sample.wav")

print("GT:", sample["text"])
print("PRED:", result["text"])

In [ ]:
from jiwer import wer

print("WER:", wer(sample["text"], result["text"]))

In [ ]:
!pip install jiwer

In [ ]:
from jiwer import wer

In [ ]:
from jiwer import wer

In [ ]:
!pip install jiwer soundfile openai-whisper datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

In [ ]:
import whisper

model = whisper.load_model("small").to("cuda")

In [ ]:
!pip install jiwer soundfile openai-whisper datasets

from datasets import load_dataset
import whisper
from jiwer import wer
import soundfile as sf

# load dataset
dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

# load model
model = whisper.load_model("small").to("cuda")

results = []
n = 10

for i in range(n):
    sample = dataset[i]

    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    sf.write(f"temp_{i}.wav", audio, sr)

    result = model.transcribe(f"temp_{i}.wav")

    gt = sample["text"].lower().strip()
    pred = result["text"].lower().strip()

    error = wer(gt, pred)

    results.append({
        "gt": gt,
        "pred": pred,
        "wer": error
    })

for r in results[:3]:
    print("\nGT:", r["gt"])
    print("PRED:", r["pred"])
    print("WER:", r["wer"])

In [ ]:
avg_wer = sum(r["wer"] for r in results) / len(results)
print("Average WER:", avg_wer)

In [ ]:
errors = [r for r in results if r["wer"] > 0]

print("Total samples:", len(results))
print("Errors:", len(errors))

In [ ]:
for e in errors[:5]:
    print("\nGT:", e["gt"])
    print("PRED:", e["pred"])

In [ ]:
def classify_error(gt, pred):
    if gt == pred:
        return "Exact Match"
    elif any(char.isdigit() for char in gt):
        return "Number Error"
    elif len(gt.split()) != len(pred.split()):
        return "Insertion/Deletion"
    else:
        return "Substitution"

for e in errors:
    e["type"] = classify_error(e["gt"], e["pred"])

from collections import Counter
print("Error Distribution:", Counter(e["type"] for e in errors))

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("asr_results.csv", index=False)

In [ ]:
!pip install jiwer soundfile openai-whisper datasets

from datasets import load_dataset
import whisper
from jiwer import wer
import soundfile as sf
import pandas as pd

# load dataset
dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

# load model
model = whisper.load_model("small").to("cuda")

results = []
n = 10

for i in range(n):
    sample = dataset[i]

    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    sf.write(f"temp_{i}.wav", audio, sr)

    result = model.transcribe(f"temp_{i}.wav")

    gt = sample["text"].lower().strip()
    pred = result["text"].lower().strip()

    error = wer(gt, pred)

    results.append({
        "gt": gt,
        "pred": pred,
        "wer": error
    })

# create dataframe
df = pd.DataFrame(results)

# save file
df.to_csv("asr_results.csv", index=False)

print("Saved successfully!")
print(df.head())

In [ ]:
from google.colab import files
files.download("asr_results.csv")

In [ ]:
print("Average WER:", df["wer"].mean())

In [ ]:
def classify_error(gt, pred):
    if gt == pred:
        return "Exact"
    elif "mr" in pred and "mister" in gt:
        return "Abbreviation"
    elif len(gt.split()) != len(pred.split()):
        return "Insertion/Deletion"
    else:
        return "Substitution"

df["error_type"] = df.apply(lambda x: classify_error(x["gt"], x["pred"]), axis=1)

print(df["error_type"].value_counts())

In [ ]:
def fix_abbreviation(text):
    return text.replace("mr.", "mister")

df["pred_fixed"] = df["pred"].apply(fix_abbreviation)

In [ ]:
from jiwer import wer

df["wer_fixed"] = df.apply(lambda x: wer(x["gt"], x["pred_fixed"]), axis=1)

print("Old WER:", df["wer"].mean())
print("New WER:", df["wer_fixed"].mean())

In [ ]:
def classify_error(gt, pred):
    if gt == pred:
        return "Exact"
    elif "mr" in pred and "mister" in gt:
        return "Abbreviation"
    elif len(gt.split()) != len(pred.split()):
        return "Insertion/Deletion"
    else:
        return "Substitution"

df["error_type"] = df.apply(lambda x: classify_error(x["gt"], x["pred"]), axis=1)

print(df["error_type"].value_counts())

In [ ]:
import re

def clean_text(text):
    return re.sub(r"[^\w\s]", "", text)

df["gt_clean"] = df["gt"].apply(clean_text)
df["pred_clean"] = df["pred_fixed"].apply(clean_text)

In [ ]:
df["wer_clean"] = df.apply(lambda x: wer(x["gt_clean"], x["pred_clean"]), axis=1)

print("WER after cleaning:", df["wer_clean"].mean())

In [ ]:
def classify_error(gt, pred):
    if gt == pred:
        return "Exact"
    elif len(gt.split()) > len(pred.split()):
        return "Deletion"
    elif len(gt.split()) < len(pred.split()):
        return "Insertion"
    else:
        return "Substitution"

df["error_type"] = df.apply(
    lambda x: classify_error(x["gt_clean"], x["pred_clean"]),
    axis=1
)

print(df["error_type"].value_counts())

In [ ]:
for i in range(5):
    print("\nGT:", df.iloc[i]["gt_clean"])
    print("PRED:", df.iloc[i]["pred_clean"])

In [ ]:
from difflib import get_close_matches

def correct_word(word, vocab):
    matches = get_close_matches(word, vocab, n=1, cutoff=0.8)
    return matches[0] if matches else word

# build vocabulary from GT
vocab = set(" ".join(df["gt_clean"]).split())

def correct_sentence(sentence):
    return " ".join([correct_word(w, vocab) for w in sentence.split()])

df["pred_corrected"] = df["pred_clean"].apply(correct_sentence)

In [ ]:
df["wer_corrected"] = df.apply(
    lambda x: wer(x["gt_clean"], x["pred_corrected"]),
    axis=1
)

print("WER after correction:", df["wer_corrected"].mean())

In [ ]:
# build vocab from a subset only (e.g., first 5 samples)
train_vocab = set(" ".join(df.iloc[:5]["gt_clean"]).split())

def correct_sentence_limited(sentence):
    from difflib import get_close_matches
    return " ".join([
        get_close_matches(w, train_vocab, n=1, cutoff=0.8)[0]
        if get_close_matches(w, train_vocab, n=1, cutoff=0.8) else w
        for w in sentence.split()
    ])

df["pred_corrected_limited"] = df["pred_clean"].apply(correct_sentence_limited)

df["wer_limited"] = df.apply(
    lambda x: wer(x["gt_clean"], x["pred_corrected_limited"]),
    axis=1
)

print("WER (no leakage):", df["wer_limited"].mean())

In [ ]:
def classify_error(gt, pred):
    if gt == pred:
        return "Exact"
    elif len(gt.split()) > len(pred.split()):
        return "Deletion"
    elif len(gt.split()) < len(pred.split()):
        return "Insertion"
    else:
        return "Substitution"

df["error_type"] = df.apply(
    lambda x: classify_error(x["gt_clean"], x["pred_clean"]),
    axis=1
)

print(df["error_type"].value_counts())

In [ ]:
print("0.21 -> 0.18 -> 0.06 -> 0.10 (real)")

In [ ]:
import matplotlib.pyplot as plt

stages = ["Baseline", "Fix", "Clean", "Final"]
values = [0.21, 0.18, 0.06, 0.10]

plt.plot(stages, values, marker='o')
plt.title("WER Improvement")
plt.ylabel("WER")
plt.show()